In [1]:
import praw
from datetime import datetime
import time
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import numpy as np

In [ ]:
reddit = praw.Reddit(user_agent=True, client_id="",
                     client_secret = "", username="", 
                     password="")

In [3]:
def unix_to_datetime(unix_time):
    """Convert Unix timestamp to datetime string."""
    return datetime.fromtimestamp(unix_time).strftime('%Y-%m-%d %H:%M:%S')

In [8]:
def pull_reddit_data(
    reddit,
    subreddit_names,
    keywords,
    min_upvotes,
    start_date,
    end_date,
    min_comment_upvotes,
    max_posts=100
):
    """
    Pull posts and comments
    Parameters:
    - reddit: PRAW Reddit instance
    - subreddit_names: List of subreddit names to search
    - keywords: List of keywords to search for in posts
    - min_upvotes: Minimum upvotes for posts
    - start_date: Unix timestamp for earliest post
    - end_date: Unix timestamp for latest post
    - min_comment_upvotes: Minimum upvotes for comments
    - max_posts: Maximum number of posts to retrieve per subreddit
    
    Returns:
    - DataFrame containing posts and comments data
    """
    # Initialize list to store data
    data = []
    
    # Convert keywords to lowercase for case-insensitive search
    keywords = [k.lower() for k in keywords]
    
    # Iterate over each subreddit
    for subreddit_name in subreddit_names:
        try:
            subreddit = reddit.subreddit(subreddit_name)
            
            # Search posts
            for submission in subreddit.search(
                ' '.join(keywords),
                sort='relevance',
                time_filter='all',
                limit=max_posts
            ):
                # Check post criteria
                post_time = submission.created_utc
                if (submission.ups >= min_upvotes and
                    start_date <= post_time <= end_date):
                    
                    # Store post data
                    post_data = {
                        'type': 'post',
                        'id': submission.id,
                        'title': submission.title,
                        'text': submission.selftext,
                        'url': submission.url,
                        'upvotes': submission.ups,
                        'created_utc': unix_to_datetime(post_time),
                        'author': submission.author.name if submission.author else '[deleted]',
                        'subreddit': submission.subreddit.display_name
                    }
                    data.append(post_data)
                    
                    # Get all comments
                    submission.comments.replace_more(limit=None)
                    for comment in submission.comments.list():
                        if comment.ups >= min_comment_upvotes:
                            comment_data = {
                                'type': 'comment',
                                'id': comment.id,
                                'title': '',
                                'text': comment.body,
                                'url': f'https://reddit.com{comment.permalink}',
                                'upvotes': comment.ups,
                                'created_utc': unix_to_datetime(comment.created_utc),
                                'author': comment.author.name if comment.author else '[deleted]',
                                'subreddit': submission.subreddit.display_name,
                                'parent_post_id': submission.id
                            }
                            data.append(comment_data)
                
                # Respect Reddit API rate limits
                time.sleep(1)
            
        except Exception as e:
            print(f"Error processing subreddit {subreddit_name}: {e}")
            continue
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    # Reorder columns
    columns = [
        'type', 'id', 'parent_post_id', 'subreddit', 'title', 'text',
        'url', 'upvotes', 'created_utc', 'author'
    ]
    # Ensure all columns exist
    for col in columns:
        if col not in df.columns:
            df[col] = ''
    
    df = df[columns]
    
    return df

In [5]:
def save_to_csv(df, filename):
    """Save DataFrame to CSV file."""
    df.to_csv(filename, index=False, encoding='utf-8')
    print(f"Data saved to {filename}")

In [49]:
if __name__ == "__main__":
    # Assuming reddit instance is already created
    # reddit = praw.Reddit(client_id='your_client_id', ...)
    
    # Example parameters
    subreddit_names = ['wallstreetbets', 'investing', 'stocks']  # Multiple subreddits
    keywords = ['AMD','Advanced Micro Devices', '$AMD']  # Keywords to search
    min_upvotes = 3  # Minimum post upvotes
    min_comment_upvotes = 3  # Minimum comment upvotes
    start_date = int(datetime(2025, 1, 1).timestamp())  # Jan 1, 2025
    end_date = int(datetime(2025, 5, 8).timestamp())  # May 1, 2025
    output_file = f'{keywords[0].upper()}.csv'
        # Pull data
    df = pull_reddit_data(
        reddit=reddit,
        subreddit_names=subreddit_names,
        keywords=keywords,
        min_upvotes=min_upvotes,
        start_date=start_date,
        end_date=end_date,
        min_comment_upvotes=min_comment_upvotes
    )
    
    # Save to CSV
    save_to_csv(df, output_file)

Data saved to AMD.csv


In [10]:
def analyze_and_compute_sentiment(df):
    """
    Perform VADER sentiment analysis and compute final sentiment scores with exponential weighting.
    
    Parameters:
    - df: DataFrame containing Reddit posts and comments
    
    Returns:
    - dict: Summary of final sentiment (weighted average compound score, categorical sentiment)
    - DataFrame: DataFrame with sentiment scores and categories
    """
    # Initialize VADER sentiment analyzer
    analyzer = SentimentIntensityAnalyzer()
    
    # Identify text column (try 'text', 'comment', 'body', or first string column)
    text_col = None
    for col in ['text', 'comment', 'body']:
        if col in df.columns:
            text_col = col
            break
    if text_col is None:
        # Fallback: use first column with string/object dtype
        for col in df.columns:
            if df[col].dtype == 'object':
                text_col = col
                break
    if text_col is None:
        raise ValueError("No suitable text column found in DataFrame")
    
    print(f"Using column '{text_col}' for sentiment analysis")
    
    # Function to get sentiment scores
    def get_sentiment_scores(text):
        if pd.isna(text) or str(text).strip() == '':
            return {'pos': 0.0, 'neu': 0.0, 'neg': 0.0, 'compound': 0.0}
        scores = analyzer.polarity_scores(str(text))
        return {
            'pos': scores['pos'],
            'neu': scores['neu'],
            'neg': scores['neg'],
            'compound': scores['compound']
        }
    
    # Apply sentiment analysis to the text column
    sentiment_scores = df[text_col].apply(get_sentiment_scores)
    
    # Add sentiment score columns
    df['sentiment_positive'] = sentiment_scores.apply(lambda x: x['pos'])
    df['sentiment_neutral'] = sentiment_scores.apply(lambda x: x['neu'])
    df['sentiment_negative'] = sentiment_scores.apply(lambda x: x['neg'])
    df['sentiment_compound'] = sentiment_scores.apply(lambda x: x['compound'])
    
    # Function to categorize sentiment based on compound score
    def get_sentiment_category(compound_score):
        if compound_score > 0.05:
            return 'Positive'
        elif compound_score < -0.05:
            return 'Negative'
        else:
            return 'Neutral'
    
    # Add sentiment category column
    df['sentiment_category'] = df['sentiment_compound'].apply(get_sentiment_category)
    
    # Compute exponential weights based on time
    def compute_exponential_weights(timestamps, decay_rate=0.1):
        """
        Compute exponential weights based on time difference from the latest timestamp.
        
        Parameters:
        - timestamps: Series of datetime strings (from 'created_utc')
        - decay_rate: Rate of exponential decay (default 0.1, ~7 days half-life)
        
        Returns:
        - Series of weights
        """
        # Convert timestamps to datetime
        dates = pd.to_datetime(timestamps)
        # Get the latest date
        latest_date = dates.max()
        # Compute time difference in days
        time_diff = (latest_date - dates).dt.total_seconds() / (24 * 3600)
        # Compute exponential weights
        weights = np.exp(-decay_rate * time_diff)
        return weights
    
    # Calculate weights
    weights = compute_exponential_weights(df['created_utc'])
    
    # Compute weighted average compound score
    avg_compound = (df['sentiment_compound'] * weights).sum() / weights.sum()
    
    # Determine overall categorical sentiment based on weighted average
    overall_category = get_sentiment_category(avg_compound)
    
    # Generate summary statistics
    summary = {
        'average_compound_score': avg_compound,
        'overall_sentiment': overall_category,
        'positive_count': len(df[df['sentiment_category'] == 'Positive']),
        'neutral_count': len(df[df['sentiment_category'] == 'Neutral']),
        'negative_count': len(df[df['sentiment_category'] == 'Negative']),
        'total_items': len(df)
    }
    
    # Optional: Breakdown by type (post/comment) and subreddit with weighted averages
    if 'type' in df.columns and 'subreddit' in df.columns:
        type_breakdown = {
            'mean': df.groupby('type').apply(
                lambda x: np.average(x['sentiment_compound'], weights=weights[x.index])
            ).to_dict(),
            'count': df.groupby('type').size().to_dict()
        }
        subreddit_breakdown = {
            'mean': df.groupby('subreddit').apply(
                lambda x: np.average(x['sentiment_compound'], weights=weights[x.index])
            ).to_dict(),
            'count': df.groupby('subreddit').size().to_dict()
        }
        summary['type_breakdown'] = type_breakdown
        summary['subreddit_breakdown'] = subreddit_breakdown
    
    return summary, df

In [12]:
def save_results(df, summary, output_file):
    """
    Save DataFrame with sentiment results and print summary.
    
    Parameters:
    - df: DataFrame with sentiment scores and categories
    - summary: Dictionary with summary statistics
    - output_file: Path to save the output CSV
    """
    # Save DataFrame to CSV
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"Updated DataFrame saved to {output_file}")
    
    # Print summary
    print("\nSentiment Analysis Summary:")
    print(f"Average Compound Score: {summary['average_compound_score']:.4f}")
    print(f"Overall Sentiment: {summary['overall_sentiment']}")
    print(f"Positive Items: {summary['positive_count']} "
          f"({summary['positive_count']/summary['total_items']*100:.1f}%)")
    print(f"Neutral Items: {summary['neutral_count']} "
          f"({summary['neutral_count']/summary['total_items']*100:.1f}%)")
    print(f"Negative Items: {summary['negative_count']} "
          f"({summary['negative_count']/summary['total_items']*100:.1f}%)")
    
    # Print breakdown if available
    if 'type_breakdown' in summary:
        print("\nSentiment by Type:")
        for type_name, stats in summary['type_breakdown']['mean'].items():
            print(f"{type_name}: Avg Compound = {stats:.4f}, Count = {summary['type_breakdown']['count'][type_name]}")
    
    if 'subreddit_breakdown' in summary:
        print("\nSentiment by Subreddit:")
        for subreddit, stats in summary['subreddit_breakdown']['mean'].items():
            print(f"{subreddit}: Avg Compound = {stats:.4f}, Count = {summary['subreddit_breakdown']['count'][subreddit]}")


In [51]:
if __name__ == "__main__":
    # Load the Reddit data CSV
    input_file = "AMD.csv"
    reddit_data = pd.read_csv(input_file)
    
    # Perform sentiment analysis and compute final sentiment
    summary, reddit_data_with_sentiment = analyze_and_compute_sentiment(reddit_data)
    
    # Define output file
    output_file = "AMD-Model"
    
    # Save results and print summary
    save_results(reddit_data_with_sentiment, summary, output_file)

Using column 'text' for sentiment analysis
Updated DataFrame saved to AMD-Model

Sentiment Analysis Summary:
Average Compound Score: 0.2123
Overall Sentiment: Positive
Positive Items: 481 (38.2%)
Neutral Items: 439 (34.9%)
Negative Items: 339 (26.9%)

Sentiment by Type:
comment: Avg Compound = 0.1787, Count = 1230
post: Avg Compound = 0.9159, Count = 29

Sentiment by Subreddit:
investing: Avg Compound = 0.1598, Count = 135
stocks: Avg Compound = 0.2135, Count = 192
wallstreetbets: Avg Compound = 0.0442, Count = 932


C:\Users\GegaMukhigulashvili\AppData\Local\Temp\ipykernel_17104\4045018566.py:109: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  'mean': df.groupby('type').apply(
C:\Users\GegaMukhigulashvili\AppData\Local\Temp\ipykernel_17104\4045018566.py:115: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  'mean': df.groupby('subreddit').apply(
